# Fine-tuning LoRA — Modèle médical expérimental

Fine-tuning de `Phi-3.5-mini-instruct` (QLoRA 4-bit) sur le dataset médical
`medical_dataset_prepared.json` (3000 paires question/réponse).

**Procédure d'exécution**
1. Activer le GPU : `Exécution` → `Modifier le type d'exécution` → GPU (T4).
2. `Exécution` → `Tout exécuter`.
3. Charger `medical_dataset_prepared.json` à l'étape 2 (fenêtre d'upload).

Sorties produites : courbe de perte, métriques d'entraînement (loss, epochs,
durée) et adaptateur LoRA exporté.


## 1. Dépendances


In [ ]:
!pip install -q -U transformers peft trl accelerate bitsandbytes datasets matplotlib


## 2. Chargement du dataset


In [ ]:
from google.colab import files
import json

uploaded = files.upload()  # medical_dataset_prepared.json
path = list(uploaded.keys())[0]
data = json.load(open(path, encoding='utf-8'))
print(f'{len(data)} paires question/réponse chargées')
print('Exemple:', data[0]['question'][:120], '...')


## 3. Mise au format Phi-3 (chat template)
Chaque paire est convertie au format de balises attendu par Phi-3.


In [ ]:
from datasets import Dataset

def to_text(ex):
    return {'text': f"<|user|>\n{ex['question']}<|end|>\n<|assistant|>\n{ex['answer']}<|end|>"}

ds = Dataset.from_list(data).map(to_text)
ds = ds.train_test_split(test_size=0.05, seed=42)  # 95% train / 5% validation
print(ds)


## 4. Chargement de Phi-3.5-mini en 4-bit (QLoRA) et configuration LoRA
La quantization 4-bit permet l'entraînement sur GPU T4 (~15 Go).

Hyperparamètres ajustables : `r` (rang LoRA, 8–32) et `EPOCHS` (1–3).


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

BASE = 'microsoft/Phi-3.5-mini-instruct'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                             device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['qkv_proj','o_proj','gate_up_proj','down_proj'])


## 5. Entraînement


In [ ]:
from trl import SFTTrainer, SFTConfig

EPOCHS = 1  # 2-3 epochs si le temps le permet

cfg = SFTConfig(
    output_dir='medical_lora',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy='steps', eval_steps=50,
    save_strategy='epoch',
    dataset_text_field='text',
    max_seq_length=1024,
    report_to='none',
)

trainer = SFTTrainer(model=model, args=cfg, peft_config=lora,
                     train_dataset=ds['train'], eval_dataset=ds['test'])
result = trainer.train()
print(result.metrics)


## 6. Métriques et courbe de perte


In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
train = [(h['step'], h['loss']) for h in hist if 'loss' in h]
val   = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]

if train: plt.plot(*zip(*train), label='train loss')
if val:   plt.plot(*zip(*val), label='val loss', marker='o')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('Courbe de perte'); plt.show()

print('Epochs:', EPOCHS)
print('Loss finale (train):', train[-1][1] if train else 'n/a')
print('Loss finale (val):', val[-1][1] if val else 'n/a')
print('Durée (s):', result.metrics.get('train_runtime'))


## 7. Test d'inférence


In [ ]:
def ask(q, max_new_tokens=160):
    prompt = f'<|user|>\n{q}<|end|>\n<|assistant|>\n'
    ids = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9)
    print(tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))

ask('What are the common symptoms of type 2 diabetes?')


## 8. Export de l'adaptateur LoRA


In [ ]:
trainer.save_model('medical_lora')
tokenizer.save_pretrained('medical_lora')
!zip -qr medical_lora.zip medical_lora
from google.colab import files
files.download('medical_lora.zip')


## Synthèse
- Modèle de base : `Phi-3.5-mini-instruct` (QLoRA 4-bit).
- Données : 3000 paires (95% train / 5% validation).
- Métriques : loss finale (train/val), nombre d'epochs, durée d'entraînement.

Modèle expérimental, non destiné à un usage clinique (cf. `medical_project/Readme.md`).
